<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/05_efficientnet_b4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 05: Clasificación con EfficientNet-B4 — Landslide4Sense

**Modelo:** EfficientNet-B4 con `timm` — soporta `in_chans=14` nativamente  
**Ventaja:** No requiere adaptar la primera capa manualmente  
**Protocolo:** Fine-tuning dos fases (freeze → unfreeze) + 5-Fold CV

![Vista previa: EfficientNet-B4 (timm, in_chans=14) — curvas de entrenamiento y F1 por fold](https://raw.githubusercontent.com/apmontesp/Landslides_-Applied-ML-Course/main/docs/figures/nb05_efficientnet_preview.png)

*Vista previa: EfficientNet-B4 (timm, in_chans=14) — curvas de entrenamiento y F1 por fold*

In [ ]:
from google.colab import drive
import os, sys, h5py, subprocess
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

# Instalación de dependencias
packages = ['timm', 'segmentation-models-pytorch', 'h5py', 'tqdm']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Montar Drive
drive.mount('/content/drive')

# Rutas de datos (misma estructura que notebooks 02-04)
base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs = list(base_path.glob('**/TrainData/img'))

if img_dirs:
    train_img_dir  = img_dirs[0]
    train_mask_dir = train_img_dir.parent / 'mask'
    DATA_ROOT  = train_img_dir.parent.parent
    img_list   = sorted(list(train_img_dir.glob('*.h5')))
    mask_list  = sorted(list(train_mask_dir.glob('*.h5')))
    print(f'Dataset: {DATA_ROOT}')
    print(f'Muestras: {len(img_list)}')
else:
    print('ERROR: No se encontro TrainData en Drive.')
    print('Verifica que la carpeta Landslide4Sense este en MyDrive.')


## 1. EfficientNet-B4 adaptado a 14 canales

In [ ]:
import torch
import timm
import torch.nn as nn

print(f'PyTorch : {torch.__version__}')
print(f'timm    : {timm.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

class EfficientNetB4Classifier(nn.Module):
    """
    EfficientNet-B4 con in_chans=14 via timm.
    timm maneja la adaptacion de la primera capa internamente.
    """
    def __init__(self, n_channels=14, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=pretrained,
            in_chans=n_channels,
            num_classes=0,        # quitar clasificador original
            global_pool='avg',
        )
        n_features = self.backbone.num_features  # 1792
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.head(self.backbone(x))

# Verificar arquitectura
model = EfficientNetB4Classifier(n_channels=14, pretrained=True).to(device)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParametros totales    : {total:,}')
print(f'Parametros entrenables: {trainable:,}')

# Test forward pass
x_test = torch.randn(2, 14, 128, 128).to(device)
with torch.no_grad():
    out = model(x_test)
print(f'Input  shape: {x_test.shape}')
print(f'Output shape: {out.shape}')
print('Forward pass OK')


## 2. Dataset y DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader

class Landslide4SenseDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.augment    = augment

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with h5py.File(self.img_paths[idx], 'r') as f:
            img = f['img'][:].astype(np.float32)    # (128,128,14)
        with h5py.File(self.mask_paths[idx], 'r') as f:
            mask = f['mask'][:].astype(np.float32)  # (128,128)

        # Z-score por canal
        for c in range(img.shape[2]):
            ch = img[:, :, c]
            img[:, :, c] = (ch - ch.mean()) / (ch.std() + 1e-8)

        label = float(mask.max() > 0)
        img_t = torch.from_numpy(img.transpose(2, 0, 1))  # (14,128,128)

        if self.augment:
            if torch.rand(1) > 0.5:
                img_t = torch.flip(img_t, dims=[2])
            if torch.rand(1) > 0.5:
                img_t = torch.flip(img_t, dims=[1])
            if torch.rand(1) > 0.5:
                img_t = torch.rot90(img_t, k=1, dims=[1, 2])

        return img_t, torch.tensor(label, dtype=torch.float32)

# Verificar carga de un batch
test_ds = Landslide4SenseDataset(img_list[:4], mask_list[:4])
img_t, lbl = test_ds[0]
print(f'Patch shape: {img_t.shape}  |  Label: {lbl.item()}')


## 3. Entrenamiento — 5-Fold Cross Validation

In [ ]:
import torch.optim as optim
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score, jaccard_score
)
import json

CFG = {
    'n_folds'      : 2,
    'epochs'       : 5,
    'freeze_epochs': 2,
    'batch_size'   : 16,   # EfficientNet-B4 es mas grande, batch menor
    'lr_head'      : 1e-4,
    'lr_backbone'  : 1e-5,
    'pos_weight'   : 0.703,
    'seed'         : 42,
}

torch.manual_seed(CFG['seed'])
output_dir = Path('/content/drive/MyDrive/Landslide4Sense/results/efficientnet_b4')
output_dir.mkdir(parents=True, exist_ok=True)

# Etiquetas completas
print('Cargando etiquetas...')
all_labels = []
for mp in tqdm(mask_list):
    with h5py.File(mp, 'r') as f:
        all_labels.append(int(f['mask'][:].max() > 0))
all_labels = np.array(all_labels)
print(f'Total: {len(all_labels)} | Positivos: {all_labels.sum()} ({all_labels.mean():.2%})')

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
fold_results = []
SEP = '='*52

for fold, (train_idx, val_idx) in enumerate(skf.split(img_list, all_labels)):
    print('\n' + SEP)
    print('FOLD ' + str(fold+1) + '/' + str(CFG['n_folds']) + '  [EfficientNet-B4]')
    print(SEP)

    train_imgs  = [img_list[i] for i in train_idx]
    train_masks = [mask_list[i] for i in train_idx]
    val_imgs    = [img_list[i] for i in val_idx]
    val_masks   = [mask_list[i] for i in val_idx]

    train_ds = Landslide4SenseDataset(train_imgs, train_masks, augment=True)
    val_ds   = Landslide4SenseDataset(val_imgs,   val_masks,   augment=False)
    train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=0, pin_memory=False)
    val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=0, pin_memory=False)

    model = EfficientNetB4Classifier(n_channels=14, pretrained=True).to(device)
    model.freeze_backbone()

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([CFG['pos_weight']]).to(device)
    )
    optimizer = optim.AdamW([
        {'params': model.head.parameters(),     'lr': CFG['lr_head']},
        {'params': model.backbone.parameters(), 'lr': CFG['lr_backbone']},
    ], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])

    history = {
        'train_loss': [], 'val_loss': [],
        'val_f1': [], 'val_auc': [],
        'val_precision': [], 'val_recall': [], 'val_iou': [],
    }
    best_f1, best_epoch = 0, 0
    last_probs, last_true = [], []

    for epoch in range(CFG['epochs']):
        if epoch == CFG['freeze_epochs']:
            model.unfreeze_backbone()
            print('  >> Backbone descongelado en epoca ' + str(epoch+1))

        # Train
        model.train()
        train_loss = 0
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs).squeeze(1), lbls)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_dl)

        # Validacion
        model.eval()
        val_loss, all_probs, all_preds, all_true = 0, [], [], []
        with torch.no_grad():
            for imgs, lbls in val_dl:
                imgs, lbls = imgs.to(device), lbls.to(device)
                logits = model(imgs).squeeze(1)
                val_loss += criterion(logits, lbls).item()
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.extend(probs.tolist())
                all_preds.extend((probs > 0.5).astype(int).tolist())
                all_true.extend(lbls.cpu().numpy().tolist())
        val_loss /= len(val_dl)

        f1   = f1_score(all_true, all_preds, zero_division=0)
        aucs = roc_auc_score(all_true, all_probs) if len(set(all_true)) > 1 else 0.0
        prec = precision_score(all_true, all_preds, zero_division=0)
        rec  = recall_score(all_true, all_preds, zero_division=0)
        iou  = jaccard_score(all_true, all_preds, zero_division=0)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(f1)
        history['val_auc'].append(aucs)
        history['val_precision'].append(prec)
        history['val_recall'].append(rec)
        history['val_iou'].append(iou)
        scheduler.step()

        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch + 1
            last_probs, last_true = all_probs, all_true
            torch.save(model.state_dict(), output_dir / f'fold{fold+1}_best.pt')

        print(
            f'  Ep {epoch+1:02d}/{CFG["epochs"]} | '
            f'loss {train_loss:.4f}/{val_loss:.4f} | '
            f'F1 {f1:.4f} | AUC {aucs:.4f} | '
            f'Prec {prec:.4f} | Rec {rec:.4f} | IoU {iou:.4f}'
        )

    fold_results.append({
        'fold': fold+1, 'best_f1': best_f1, 'best_epoch': best_epoch,
        'history': history, 'all_probs': last_probs,
        'all_true': np.array(last_true),
    })
    print('  Mejor F1: ' + f'{best_f1:.4f}' + ' (epoca ' + str(best_epoch) + ')')

summary = {
    'model'  : 'efficientnet_b4',
    'config' : CFG,
    'folds'  : [{'fold': r['fold'], 'best_f1': r['best_f1'], 'best_epoch': r['best_epoch']} for r in fold_results],
    'mean_f1': float(np.mean([r['best_f1'] for r in fold_results])),
    'std_f1' : float(np.std([r['best_f1']  for r in fold_results])),
}
with open(output_dir / 'kfold_summary.json', 'w') as fout:
    json.dump(summary, fout, indent=2)

print('\n' + SEP)
print('EFFICIENTNET-B4 COMPLETADO')
print(f'F1 medio: {summary["mean_f1"]:.4f} +/- {summary["std_f1"]:.4f}')
print(SEP)


## 4. Visualización de resultados

In [ ]:
from sklearn.metrics import (
    precision_recall_curve, roc_curve, auc,
    confusion_matrix, ConfusionMatrixDisplay
)

n_folds = len(fold_results)
fig, axes = plt.subplots(n_folds, 4, figsize=(22, 5*n_folds))
if n_folds == 1: axes = [axes]

summary_rows = []
for i, r in enumerate(fold_results):
    h = r['history']
    epochs = range(1, len(h['train_loss']) + 1)

    # 1. Grafico de Loss
    axes[i][0].plot(epochs, h['train_loss'], label='Train', color='steelblue')
    axes[i][0].plot(epochs, h['val_loss'],   label='Val',   color='orange')
    axes[i][0].set_title(f'Fold {r["fold"]} — Loss')
    axes[i][0].legend()
    axes[i][0].grid(alpha=0.3)

    # 2. Grafico de Metricas
    metrics_list = [
        ('val_f1', 'F1', 'green', '-'),
        ('val_auc', 'AUC-ROC', 'orange', '-'),
        ('val_precision', 'Precision', 'royalblue', '--'),
        ('val_recall', 'Recall', 'red', '--'),
        ('val_iou', 'IoU', 'purple', ':')
    ]
    for key, label, col, style in metrics_list:
        if key in h and len(h[key]) == len(epochs):
            axes[i][1].plot(epochs, h[key], label=label, color=col, linestyle=style)

    axes[i][1].axvline(r['best_epoch'], color='black', linestyle=':', alpha=0.6, label='Best')
    axes[i][1].set_title(f'Fold {r["fold"]} — Metricas')
    axes[i][1].legend(fontsize=7)

    # 3. Curva PR
    prec_c, rec_c, _ = precision_recall_curve(r['all_true'], r['all_probs'])
    axes[i][2].plot(rec_c, prec_c, color='darkorange', lw=2, label=f'AUC-PR={auc(rec_c, prec_c):.3f}')
    axes[i][2].set_xlabel('Recall')
    axes[i][2].set_ylabel('Precision')
    axes[i][2].set_title(f'Fold {r["fold"]} — Curva PR')
    axes[i][2].legend()

    # 4. Matriz de Confusion
    preds = (np.array(r['all_probs']) > 0.5).astype(int)
    cm = confusion_matrix(r['all_true'], preds)
    ConfusionMatrixDisplay(cm, display_labels=['Estable', 'Desliz.']).plot(cmap='YlOrRd', ax=axes[i][3], colorbar=False)

    summary_rows.append({
        'Fold': r['fold'], 'F1': r['best_f1'], 'AUC': h['val_auc'][r['best_epoch']-1], 
        'Prec': h['val_precision'][r['best_epoch']-1], 'Recall': h['val_recall'][r['best_epoch']-1], 
        'IoU': h['val_iou'][r['best_epoch']-1], 'Ep': r['best_epoch']
    })

plt.tight_layout()
plt.show()

print('\nRESUMEN POR FOLD')
print('-'*65)
print(f'{"Fold":>4} | {"F1":>6} | {"AUC":>6} | {"Prec":>6} | {"Recall":>6} | {"IoU":>6} | Ep.')
print('-'*65)
for row in summary_rows:
    print(f'{row["Fold"]:>4} | {row["F1"]:>6.4f} | {row["AUC"]:>6.4f} | '
          f'{row["Prec"]:>6.4f} | {row["Recall"]:>6.4f} | {row["IoU"]:>6.4f} | {row["Ep"]}')
print('-'*65)
means = {k: np.mean([r[k] for r in summary_rows]) for k in ['F1','AUC','Prec','Recall','IoU']}
print(f'{"MEAN":>4} | {means["F1"]:>6.4f} | {means["AUC"]:>6.4f} | '
      f'{means["Prec"]:>6.4f} | {means["Recall"]:>6.4f} | {means["IoU"]:>6.4f}')


## Resumen

Los pesos del mejor modelo de cada fold se guardan en Drive:  
`MyDrive/Landslide4Sense/results/efficientnet_b4/fold{N}_best.pt`

Siguiente paso → `06_unet_segmentation.ipynb`